# Fase 4 — Análisis de desempeño y reentrenamiento
## IgnisAI · Proyecto de Datos I · UCM

Este notebook cubre los apartados **1.1, 1.3 y 1.4** del enunciado de la Fase 4:

- **1.1** Reentrenar el mejor modelo con todos los datos de fase 3
- **1.3** Evaluar el modelo sobre los nuevos datos de 2026 y analizar diferencias
- **1.4** (Opcional) Reentrenar con todos los datos disponibles (fase 3 + 2026)

Todos los datos se cargan directamente desde MinIO. No se necesita nada en local.


## 0. Imports y configuración

In [ ]:
import sys
sys.path.append("src")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    fbeta_score, precision_score, recall_score, roc_auc_score,
    ConfusionMatrixDisplay
)
from extraccion import minioFunctions as mf
from modelos.utils.metricas import evaluar_clasificacion

# Colores del proyecto
COLOR_FASE3 = "#2563EB"
COLOR_NUEVO = "#DC2626"
COLOR_COMB  = "#16A34A"

# Hiperparámetros del mejor run (sweep: 3pdmt88q, run: v5cmrozh)
MEJORES_HIPERPARAMETROS = {
    "n_estimators":    2000,
    "learning_rate":   0.17971228299068043,
    "max_depth":       4,
    "subsample":       0.8142987782803428,
    "colsample_bytree": 0.8311621901655458,
    "min_child_weight": 1,
    "gamma":           0,
}
UMBRAL = 0.35189353387841105
SEED   = 42
COLS_ELIMINAR    = ["porcentaje", "temp_max", "temp_min", "NDVI", "pressure_mean"]
COLS_NO_FEATURES = ["final", "date", "_year"]

# Métricas de test de fase 3 (de la memoria)
METRICAS_FASE3_MEMORIA = {
    "accuracy":  0.9547,
    "f1":        0.3724,
    "f2":        0.3775,
    "precision": 0.3641,
    "recall":    0.3810,
}

print("✅ Imports OK")


## 1. Carga de datos desde MinIO

In [ ]:
cliente = mf.crear_cliente()

# Dataset fase 3
print("Descargando MINI.parquet (fase 3)...")
df_fase3 = mf.bajar_fichero(cliente, "grupo3/cleaned/MINI.parquet")
df_fase3["date"] = pd.to_datetime(df_fase3["date"])
print(f"  → {df_fase3.shape[0]:,} registros | {df_fase3['final'].sum():,} incendios ({df_fase3['final'].mean()*100:.2f}%)")

# Dataset nuevos 2026
print("\nDescargando final_cleaned_2026.parquet...")
df_nuevos = mf.bajar_fichero(cliente, "grupo3/cleaned/final_cleaned_2026.parquet")
df_nuevos["date"] = pd.to_datetime(df_nuevos["date"])
print(f"  → {df_nuevos.shape[0]:,} registros | {df_nuevos['final'].sum():,} incendios ({df_nuevos['final'].mean()*100:.2f}%)")

print("\n✅ Datos cargados correctamente")


## 2. Preparación de features

In [ ]:
def preparar_X_y(df):
    df2 = df.drop(columns=COLS_ELIMINAR, errors="ignore")
    y = df2["final"]
    X = df2.drop(columns=[c for c in COLS_NO_FEATURES if c in df2.columns], errors="ignore")
    return X, y

X_fase3, y_fase3 = preparar_X_y(df_fase3)
X_nuevos, y_nuevos = preparar_X_y(df_nuevos)

# Alinear columnas
cols_comunes = [c for c in X_fase3.columns if c in X_nuevos.columns]
X_fase3  = X_fase3[cols_comunes]
X_nuevos = X_nuevos[cols_comunes]

print(f"Features usadas ({len(cols_comunes)}): {cols_comunes}")


## 3. Apartado 1.1 — Reentrenamiento con todos los datos de fase 3

Se reentrena el mejor modelo XGBoost usando **todos los datos de fase 3** (train + val + test).
Este es el modelo que pasa a producción antes de incorporar los datos nuevos.


In [ ]:
ratio_fase3 = (y_fase3 == 0).sum() / (y_fase3 == 1).sum()
print(f"Ratio de clases (scale_pos_weight): {ratio_fase3:.2f}")

clf_fase3 = xgb.XGBClassifier(
    **MEJORES_HIPERPARAMETROS,
    scale_pos_weight=ratio_fase3,
    random_state=SEED,
    eval_metric="aucpr",
    n_jobs=-1,
)

print("Entrenando con todos los datos de fase 3...")
clf_fase3.fit(X_fase3, y_fase3)
print("✅ Entrenamiento completado")


## 4. Apartado 1.3 — Evaluación sobre los nuevos datos de 2026

Se evalúa el modelo entrenado con fase 3 sobre los **550 registros nuevos de 2026**
que no participaron en el entrenamiento. Esta es la evaluación real del modelo en producción.


In [ ]:
def evaluar_modelo(clf, X, y, nombre, umbral=UMBRAL):
    y_prob = clf.predict_proba(X)[:, 1]
    y_pred = (y_prob >= umbral).astype(int)
    metricas = {
        "accuracy":  round(accuracy_score(y, y_pred), 4),
        "f1":        round(f1_score(y, y_pred, zero_division=0), 4),
        "f2":        round(fbeta_score(y, y_pred, beta=2, zero_division=0), 4),
        "precision": round(precision_score(y, y_pred, zero_division=0), 4),
        "recall":    round(recall_score(y, y_pred, zero_division=0), 4),
        "auc_roc":   round(roc_auc_score(y, y_prob), 4),
    }
    cm = confusion_matrix(y, y_pred)
    print(f"\n{'='*50}")
    print(f"MÉTRICAS — {nombre}")
    print(f"{'='*50}")
    for k, v in metricas.items():
        print(f"  {k:<12}: {v}")
    print(f"\n  Matriz de confusión:")
    print(f"    TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"    FN={cm[1,0]}  TP={cm[1,1]}")
    return metricas, cm, y_prob, y_pred

metricas_nuevos, cm_nuevos, y_prob_nuevos, y_pred_nuevos = evaluar_modelo(
    clf_fase3, X_nuevos, y_nuevos, "Modelo fase 3 — datos nuevos 2026"
)


### 4.1 Comparativa con métricas de test de fase 3

In [ ]:
metricas_labels = ["accuracy", "f1", "f2", "precision", "recall"]
v_mem   = [METRICAS_FASE3_MEMORIA[m] for m in metricas_labels]
v_nuevo = [metricas_nuevos[m] for m in metricas_labels]

print(f"{'Métrica':<12} {'Fase 3 (test)':>14} {'Datos 2026':>12} {'Diferencia':>12}")
print("-" * 52)
for k, v3, vn in zip(metricas_labels, v_mem, v_nuevo):
    diff = round(vn - v3, 4)
    signo = "+" if diff > 0 else ""
    print(f"{k:<12} {v3:>14} {vn:>12} {signo}{diff:>11}")

# Gráfica
x = np.arange(len(metricas_labels))
w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparativa de métricas y matriz de confusión", fontsize=13, fontweight="bold")

ax = axes[0]
b3 = ax.bar(x - w/2, v_mem,   width=w, label="Fase 3 (test)",   color=COLOR_FASE3, alpha=0.8)
bn = ax.bar(x + w/2, v_nuevo, width=w, label="Datos 2026",       color=COLOR_NUEVO, alpha=0.8)
for bar in list(b3) + list(bn):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(metricas_labels)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Valor")
ax.set_title("Métricas: Fase 3 vs. Datos nuevos 2026")
ax.legend()

ax = axes[1]
disp = ConfusionMatrixDisplay(confusion_matrix=cm_nuevos, display_labels=["No incendio", "Incendio"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Matriz de confusión — datos 2026")

plt.tight_layout()
plt.savefig("resultados_fase4_metricas.png", dpi=150, bbox_inches="tight")
plt.show()


### 4.2 Análisis de distribuciones — Variable objetivo y variables clave

El enunciado pide analizar la variable objetivo y las 3 variables más importantes.
Según los sweeps de fase 3, las más relevantes son: **NDWI, dist_civ, humidity_mean**.


In [ ]:
VARS_ANALIZAR = ["NDWI", "dist_civ", "humidity_mean"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Distribución: Fase 3 vs. Datos nuevos 2026", fontsize=13, fontweight="bold")
axes_flat = axes.flatten()

# Variable objetivo
ax = axes_flat[0]
labels = ["No incendio (0)", "Incendio (1)"]
f3_counts = [df_fase3["final"].value_counts().get(0,0), df_fase3["final"].value_counts().get(1,0)]
n26_counts = [df_nuevos["final"].value_counts().get(0,0), df_nuevos["final"].value_counts().get(1,0)]
f3_pct  = [c/sum(f3_counts)*100 for c in f3_counts]
n26_pct = [c/sum(n26_counts)*100 for c in n26_counts]
x = np.arange(2)
w = 0.35
ax.bar(x - w/2, f3_pct,  width=w, label="Fase 3",       color=COLOR_FASE3, alpha=0.8)
ax.bar(x + w/2, n26_pct, width=w, label="Datos 2026",   color=COLOR_NUEVO, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("% del dataset")
ax.set_title("Variable objetivo (final)")
ax.legend()
for i, (p3, pn) in enumerate(zip(f3_pct, n26_pct)):
    ax.text(i - w/2, p3 + 0.5, f"{p3:.1f}%", ha="center", fontsize=8)
    ax.text(i + w/2, pn + 0.5, f"{pn:.1f}%", ha="center", fontsize=8)

# Variables clave
for i, var in enumerate(VARS_ANALIZAR, 1):
    ax = axes_flat[i]
    v3  = df_fase3[var].dropna()
    vn  = df_nuevos[var].dropna()
    xmin = min(v3.quantile(0.01), vn.quantile(0.01))
    xmax = max(v3.quantile(0.99), vn.quantile(0.99))
    bins = np.linspace(xmin, xmax, 40)
    ax.hist(v3, bins=bins, color=COLOR_FASE3, alpha=0.6, density=True, label=f"Fase 3 (μ={v3.mean():.2f})")
    ax.hist(vn, bins=bins, color=COLOR_NUEVO, alpha=0.6, density=True, label=f"2026 (μ={vn.mean():.2f})")
    ax.axvline(v3.mean(), color=COLOR_FASE3, linestyle="--", linewidth=1.5)
    ax.axvline(vn.mean(), color=COLOR_NUEVO, linestyle="--", linewidth=1.5)
    ax.set_title(var, fontweight="bold")
    ax.set_ylabel("Densidad")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("resultados_fase4_distribuciones.png", dpi=150, bbox_inches="tight")
plt.show()

# Tabla estadística
print("\nEstadísticos comparativos:")
print(f"{'Variable':<15} {'Fase3 media':>12} {'2026 media':>12} {'Fase3 std':>12} {'2026 std':>12}")
print("-" * 65)
for var in ["final"] + VARS_ANALIZAR:
    v3 = df_fase3[var]
    vn = df_nuevos[var]
    print(f"{var:<15} {v3.mean():>12.3f} {vn.mean():>12.3f} {v3.std():>12.3f} {vn.std():>12.3f}")


### 4.3 Conclusiones del análisis de desempeño

**Diferencia en métricas:**
El modelo entrenado con datos de fase 3 obtiene un **F2=0.38 en test** (fase 3).
Al evaluarlo sobre los nuevos datos de 2026, el F2 baja a **0.13**.

**Causas de la degradación (data drift estacional):**
- **NDWI**: sube de -0.44 a -0.34 — mayor humedad en invierno/primavera
- **dist_civ**: sube de 32 km a 84 km — los datos de 2026 incluyen zonas más remotas, fuera de la distribución de entrenamiento
- **humidity_mean**: también varía, coherente con la época del año

Los datos nuevos corresponden a **enero–abril de 2026**, período con muy pocos incendios (22 de 550 registros, 4%). El modelo fue entrenado mayoritariamente con datos de verano, cuando los incendios tienen un perfil meteorológico diferente.

**El AUC-ROC se mantiene en 0.81**, lo que indica que el modelo sigue siendo capaz de discriminar entre clases — el problema es la calibración del umbral, no la capacidad predictiva en sí.


## 5. Apartado 1.4 (Opcional) — Reentrenamiento con fase 3 + datos 2026

Se reentrena el modelo incorporando los 550 nuevos registros de 2026 al dataset de entrenamiento.


In [ ]:
# Combinar datasets
X_total = pd.concat([X_fase3, X_nuevos], ignore_index=True)
y_total = pd.concat([y_fase3, y_nuevos], ignore_index=True)

print(f"Dataset combinado: {X_total.shape[0]:,} registros | {y_total.sum():,} incendios ({y_total.mean()*100:.2f}%)")

ratio_total = (y_total == 0).sum() / (y_total == 1).sum()
print(f"Ratio de clases: {ratio_total:.2f}")

clf_total = xgb.XGBClassifier(
    **MEJORES_HIPERPARAMETROS,
    scale_pos_weight=ratio_total,
    random_state=SEED,
    eval_metric="aucpr",
    n_jobs=-1,
)

print("\nEntrenando con datos fase 3 + 2026...")
clf_total.fit(X_total, y_total)
print("✅ Entrenamiento completado")


In [ ]:
metricas_total, cm_total, _, _ = evaluar_modelo(
    clf_total, X_nuevos, y_nuevos, "Modelo reentrenado (fase3 + 2026) — datos 2026"
)

# Comparativa de los tres escenarios
print(f"\n{'='*65}")
print("RESUMEN COMPARATIVO — Evaluación sobre datos nuevos 2026")
print(f"{'='*65}")
print(f"{'Métrica':<12} {'Fase3 (memoria)':>16} {'Modelo F3→2026':>16} {'Modelo F3+2026':>16}")
print("-" * 62)
for k in ["accuracy", "f1", "f2", "precision", "recall"]:
    vm = METRICAS_FASE3_MEMORIA[k]
    vn = metricas_nuevos[k]
    vt = metricas_total[k]
    print(f"{k:<12} {vm:>16} {vn:>16} {vt:>16}")


## 6. Guardado del modelo final en MinIO

In [ ]:
# Guardar modelo reentrenado con fase3 + 2026 (el que va a producción)
joblib.dump(clf_total, "xgboost_clasificacion_2026.pkl")

# Subir a MinIO usando put_object directamente (pkl no es parquet)
import io
with open("xgboost_clasificacion_2026.pkl", "rb") as f:
    datos = f.read()

buffer = io.BytesIO(datos)
cliente.put_object(
    bucket_name="pd1",
    object_name="grupo3/modelos/xgboost_clasificacion_2026.pkl",
    data=buffer,
    length=len(datos)
)
print("✅ Modelo subido a MinIO: grupo3/modelos/xgboost_clasificacion_2026.pkl")
